# Exploração Base de Dados

In [ ]:
# Importar Bibliotecas 

# Carregar Imagem
import os
from datasets import load_dataset
from tqdm import tqdm 

# Exploração dos Dados
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
# Carrega o dataset na memória
dataset = load_dataset("dragonintelligence/CIFAKE-image-dataset")

In [ ]:
# Define o diretório de destino local
base_dir = "../data/raw" 

In [ ]:
# Função para estruturar e salvar as imagens nas pastas corretas
def export_images_to_local(hf_dataset, split_name):
    print(f"\nIniciando exportação estruturada do conjunto de {split_name.upper()}...")
    
    # A barra de progresso agora será em modo texto (seguro)
    for i, item in enumerate(tqdm(hf_dataset[split_name])):
        image = item['image']
        label = item['label']
        
        # Mapeamento estrito da classe: 0 = real (CIFAR-10), 1 = fake (Stable Diffusion)
        class_name = "real" if label == 0 else "fake"
        
        # Cria a hierarquia: ../data/raw/train/real, ../data/raw/train/fake, etc.
        save_dir = os.path.join(base_dir, split_name, class_name)
        os.makedirs(save_dir, exist_ok=True)
        
        # Salva a imagem em PNG para preservar intactos os micro-artefatos de difusão
        image_path = os.path.join(save_dir, f"{class_name}_{i}.png")
        image.save(image_path)

In [ ]:
print("Baixando o dataset CIFAKE do Hugging Face (Isso pode levar alguns minutos)...")
# Executa a extração
export_images_to_local(dataset, 'train')
export_images_to_local(dataset, 'test')

print("\nAs 120.000 imagens foram exportadas com sucesso para a pasta data/raw/!")

In [ ]:
# Definir caminhos locais para as pastas de treino
caminho_real = '../data/raw/train/real'
caminho_fake = '../data/raw/train/fake'

In [ ]:
# Mapear o caminho das 5 primeiras imagens .png de cada classe
arquivos_reais = [os.path.join(caminho_real, f) for f in os.listdir(caminho_real) if f.endswith('.png')][:5]
arquivos_fakes = [os.path.join(caminho_fake, f) for f in os.listdir(caminho_fake) if f.endswith('.png')][:5]

In [ ]:
# Carregar imagens com OpenCV e converter de BGR para RGB
def carregar_imagem_cv2(caminho):
    # cv2.imread retorna um array NumPy no formato BGR
    img_bgr = cv2.imread(caminho)
    # Converter para RGB para compatibilidade com a visualização no Matplotlib
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb

reais = [carregar_imagem_cv2(arq) for arq in arquivos_reais]
fakes = [carregar_imagem_cv2(arq) for arq in arquivos_fakes]

In [ ]:
# Visualização: Comparativo Visual Real vs. Sintético
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("CIFAKE (32x32): Imagens Reais (CIFAR-10) vs Sintéticas (Stable Diffusion) via OpenCV", fontsize=14)

for i in range(5):
    # Plotando imagens reais
    axes[0, i].imshow(reais[i])
    axes[0, i].set_title("Real")
    axes[0, i].axis('off')
    
    # Plotando imagens sintéticas
    axes[1, i].imshow(fakes[i])
    axes[1, i].set_title("Sintética")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Exploração Estatística: Histogramas de Canais utilizando cv2.calcHist
canais = ('Red', 'Green', 'Blue')
cores = ('r', 'g', 'b')

# Adição do parâmetro sharey=True para unificar a escala do eixo Y
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
fig.suptitle("Distribuição Estatística de Pixels (Canais RGB)", fontsize=14)

for i, cor in enumerate(cores):
    # Computando Histograma da primeira imagem Real com OpenCV
    # cv2.calcHist([imagem], [canal], mascara, [tamanho_bins], [intervalo])
    histograma_real = cv2.calcHist([reais[0]], [i], None, [256], [0, 256])
    axes[0].plot(histograma_real, color=cor, alpha=0.7)
    
    # Computando Histograma da primeira imagem Sintética com OpenCV
    histograma_fake = cv2.calcHist([fakes[0]], [i], None, [256], [0, 256])
    axes[1].plot(histograma_fake, color=cor, alpha=0.7)

# Configurações do eixo da Imagem Real
axes[0].set_title("Distribuição - Imagem Real")
axes[0].set_xlabel("Intensidade do Pixel (0-255)")
axes[0].set_ylabel("Frequência")
axes[0].set_xlim([0, 256])

# Configurações do eixo da Imagem Sintética
axes[1].set_title("Distribuição - Imagem Sintética")
axes[1].set_xlabel("Intensidade do Pixel (0-255)")
# Ocultando o ylabel do segundo gráfico, pois o eixo Y agora é compartilhado
axes[1].set_xlim([0, 256])

plt.tight_layout()
plt.show()

### 📌 Conclusões da Análise de Distribuição de Frequência (RGB)

A inspeção visual dos histogramas de intensidade dos canais de cor (Red, Green, Blue) revela divergências estatísticas fundamentais entre as amostras reais e as geradas por Inteligência Artificial, corroborando a literatura forense sobre Modelos de Difusão Latente:

* **Concentração Espectral em Imagens Reais (CIFAR-10):** A distribuição dos pixels nas imagens autênticas apresenta um comportamento denso e concentrado na metade inferior e central da escala (intensidades entre 0 e 150). Nota-se uma ausência virtual de pixels com alta luminosidade, com a frequência caindo a zero após a marca de 180.
* **Uniformidade e Dispersão em Imagens Sintéticas (Stable Diffusion):** A imagem artificial exibe uma distribuição consideravelmente mais uniforme e espalhada por todo o espectro (0 a 255). Os picos de frequência são mais baixos, porém constantes, indicando uma variação artificialmente ampla das intensidades cromáticas.
* **Piso de Ruído em Alta Intensidade:** O principal diferencial observado é a presença significativa de pixels na extremidade direita do eixo X (intensidades entre 200 e 255) para a imagem sintética, faixa na qual a imagem real é nula. 

**Insight Metodológico:**
Esse espalhamento artificial e a presença de picos em altas intensidades corroboram a hipótese de que o processo de decodificação e *upsampling* do *Stable Diffusion* introduz um "piso de ruído" (*noise floor*) característico em toda a imagem. Isso confirma a premissa do projeto de que o conjunto CIFAKE preserva micro-ruídos cruciais e que a detecção não deve focar em anomalias semânticas, mas sim nestes artefatos estatísticos de alta frequência invisíveis a olho nu, mas capturáveis pelos filtros da rede All-CNN-C.

In [ ]:
# Exploração Espectral: Transformada Rápida de Fourier (FFT)

# Convertendo uma imagem de cada classe para escala de cinza (2D)
# A biblioteca cv2 já está importada do bloco anterior, assim como as listas reais e fakes.
img_real_gray = cv2.cvtColor(reais[1], cv2.COLOR_RGB2GRAY)
img_fake_gray = cv2.cvtColor(fakes[1], cv2.COLOR_RGB2GRAY)

In [ ]:
def computar_espectro_fourier(imagem_gray):
    # Computa a Transformada de Fourier 2D
    f_transform = np.fft.fft2(imagem_gray)
    
    # Desloca o componente de frequência zero (DC) para o centro do espectro
    f_shift = np.fft.fftshift(f_transform)
    
    # Calcula o espectro de magnitude em escala logarítmica para visualização 
    espectro_magnitude = 20 * np.log(np.abs(f_shift) + 1)
    return espectro_magnitude

In [ ]:
espectro_real = computar_espectro_fourier(img_real_gray)
espectro_fake = computar_espectro_fourier(img_fake_gray)

In [ ]:
# Visualização dos Espectros de Frequência
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Análise no Domínio da Frequência (Espectro de Magnitude FFT)", fontsize=14)

# Plot Espectro Real
axes[0].imshow(espectro_real, cmap='gray')
axes[0].set_title("Espectro de Fourier - Imagem Real")
axes[0].axis('off')

# Plot Espectro Sintético
axes[1].imshow(espectro_fake, cmap='gray')
axes[1].set_title("Espectro de Fourier - Imagem Sintética")
axes[1].axis('off')

plt.tight_layout()
plt.show()

### ⚠️ Resultados Inconclusivos da Inspeção Visual no Domínio da Frequência (FFT)

A tentativa de diagnosticar a origem da imagem a partir da análise visual pura dos espectros gerados pela Transformada Rápida de Fourier (FFT) demonstrou resultados inconclusivos para a percepção humana. Esse cenário empírico reforça a relevância da abordagem computacional e algorítmica proposta neste projeto forense.

**Fundamentação Teórica das Limitações Visuais:**
* Investigações profundas apontam que as imagens sintéticas exibem discrepâncias severas no domínio espectral de alta frequência (quando analisadas sob a Transformada Rápida de Fourier).
* Contudo, as anomalias estruturais e o "piso de ruído" residual (noise floor) são inseridos no tecido da imagem de forma sub-perceptível.
* As restrições dimensionais severas impostas pelo CIFAKE forçam o modelo preditivo a descartar grande parte do contexto semântico macroscópico.
* A investigação interativa acerca da robustez para a detecção de geração sintética transcende invariavelmente os moldes ingênuos das intuições semânticas primitivas macro-visuais humanas.
* A literatura forense condena avaliações explicativas pautadas meramente na inspeção visual humana subjetiva, visto que o sistema cognitivo humano é inerentemente enviesado.